# 第2章　自动微分（Autograd）— 学习的心脏

神经网络的"学习"就是**朝着让误差（损失）变小的方向，一点点调整参数**。
指出那个"方向"的就是**梯度（gradient）**，PyTorch 会自动计算它，这就是 Autograd。

本章目标：能解释 `backward()` 在做什么，并**亲手实现梯度下降**。

> **本笔记使用方法**
> - 从上到下依次运行单元格（Colab/Jupyter 都是 `Shift + Enter`）。
> - 代码**稍作修改、弄坏再修好**最能进步。每章末尾有练习。
> - 多数章节不需要 GPU。较重的章节（CNN）会说明用法。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 2-0. 数学小复习（忘了没关系，在这里想起来）

- **求导＝斜率**：函数 $f(x)$ 的导数 $f'(x)$ 表示"x 稍微动一点时 f 变化多少"。斜率为正则上升，为负则下降。
- **朝最小值移动**：想让 $f$ 变小，就朝斜率的**反方向**移动 $x$。
  $$x \leftarrow x - \eta\, f'(x)$$
  这里 $\eta$（eta）是**学习率**＝一步的大小。这就是**梯度下降法**。
- **梯度（gradient）**：当变量有多个（$w_1, w_2, \dots$）时，"各方向斜率"的集合 $\nabla f$。
- **链式法则（chain rule）**：复合函数求导。$y=f(g(x))$ 则 $\dfrac{dy}{dx}=\dfrac{dy}{dg}\cdot\dfrac{dg}{dx}$。
  神经网络是函数的嵌套，所以把链式法则从末端依次相乘即可＝**误差反向传播（backprop）**。PyTorch 自动完成。

## 2-1. `requires_grad` 与 `backward()`

加了 `requires_grad=True` 的张量会成为"追踪对象"，用它做的运算会被记录。
在最终值上调用 `.backward()`，各变量的 `.grad` 里就会得到梯度。

In [ ]:
import torch

w = torch.tensor(3.0, requires_grad=True)   # 追踪梯度
y = w ** 2                                  # y = w^2
y.backward()                                # 计算 dy/dw
print("w.grad =", w.grad)                   # 2*w = 6.0（与手算一致）

手算：$y=w^2$ 则 $\dfrac{dy}{dw}=2w$。$w=3$ 时为 $6$，与 PyTorch 的答案一致。

### 链式法则的例子
$y = (2w+1)^2$ 在 $w=1$。手算：$\frac{dy}{dw}=2(2w+1)\cdot 2 = 4(2w+1) = 12$。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
u = 2 * w + 1
y = u ** 2
y.backward()
print("w.grad =", w.grad)   # 12.0

## 2-2. "亲手"实现梯度下降（最重要・直觉的核心）

求让 $f(x) = (x-3)^2$ 最小的 $x$。答案显然是 $x=3$，但我们体验**只靠梯度一点点逼近**的过程。
这就是所有训练的缩影。

In [ ]:
x = torch.tensor(0.0, requires_grad=True)   # 从随意的初始值出发
lr = 0.1                                      # 学习率（一步的大小）

for step in range(20):
    f = (x - 3) ** 2          # 想最小化的函数（当作损失）
    f.backward()              # 计算 df/dx -> 进入 x.grad

    with torch.no_grad():     # 更新不追踪（下面说明）
        x -= lr * x.grad      # 朝梯度反方向走一步
    x.grad.zero_()            # 为下次清零梯度（重要！）

    if step % 4 == 0:
        print(f"step {step:2d}: x={x.item():.4f}, f={f.item():.4f}")

print("最终 x =", x.item(), "（应接近 3）")

上面循环里的内容，和后面章节的"训练循环"几乎一样：
**① 算损失 → ② `backward()` 求梯度 → ③ 朝梯度反方向更新 → ④ 清零梯度**。

## 2-3. `torch.no_grad()` 与 `detach()`：停止追踪

- 把"乘以学习率更新参数"也纳入求导既麻烦又浪费，所以用 `with torch.no_grad():` 包起来。
- 推理（预测）时也不需要梯度，用 `no_grad` 包起来更**快、更省内存**。
- 只想从计算图里取出数值时用 `.detach()`。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

with torch.no_grad():
    z = x * 5            # 这个计算不被追踪
print("z.requires_grad =", z.requires_grad)   # False

val = y.detach()         # 从 y 切断梯度联系后的数值
print("detach:", val, val.requires_grad)

## 2-4. 为什么每次都要 `zero_()`（清零梯度）

PyTorch 每次 `.backward()` 都会把梯度 **累加**。不清零的话上一次的梯度会残留，训练就坏了。
（后面章节由 `optimizer.zero_grad()` 承担这个职责。）

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = w * 2          # dy/dw = 2
    y.backward()
    print(f"不清零 {i}: w.grad = {w.grad.item()}")  # 2,4,6 不断累加！

print("--- 正确做法是每次清零 ---")
w.grad.zero_()
for i in range(3):
    y = w * 2
    y.backward()
    print(f"清零 {i}: w.grad = {w.grad.item()}")   # 始终是 2
    w.grad.zero_()

## 练习 2
1. 用梯度下降最小化 $f(x)=(x+5)^2$，确认 $x$ 逼近 $-5$。
2. 把学习率 `lr` 改成 `0.01`、`0.9`、`1.1` 会怎样？（太小=慢，太大=发散）
3. 求 `y = w**3` 在 `w=2` 的梯度，确认与手算 $3w^2=12$ 一致。

In [ ]:
# 在这里写你自己的代码并运行
